# Inference Timing Benchmark

Benchmark FCN, LSTM, and Transformer inference speed with:
- Batch size = 1
- Warm-up cycles
- CPU and CUDA devices
- Separate timing for dataloader, `to(device)`, and forward pass

In [ ]:
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch.utils.data import DataLoader, ConcatDataset
from tqdm.auto import tqdm

import shaft_force_sensing.models
from shaft_force_sensing.models import LitLSTM
from shaft_force_sensing.training import prepare_test_dataset, load_model

%load_ext autoreload
%autoreload 2

## Config

In [ ]:
DATA_ROOT = Path("../data")
CKPT_ROOT = Path("../logs")

CHECKPOINT_DIRS = {
    "transformer": CKPT_ROOT / "transformer" / "base",
    "lstm": CKPT_ROOT / "lstm" / "base",
    "fcn": CKPT_ROOT / "fcn" / "base",
}

TELEOP = False
MODEL_IDX = 0
MAX_STEPS = 1100
WARMUP_STEPS = 100

BATCH_SIZE = 1
NUM_WORKERS = 0
PIN_MEMORY = True

## Helpers

In [ ]:
def maybe_sync(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def move_batch_to_device(batch, device: torch.device):
    return tuple(t.to(device, non_blocking=(device.type == "cuda")) for t in batch)


def run_model_test_step(model, batch, batch_idx: int):
    # Use model's test_step path for timing consistency with evaluation logic.
    return model.test_step(batch, batch_idx)


def build_loader_for_model(model, data_root: Path, teleop: bool, model_idx: int):
    model_name = model._get_name()
    test_sets, _ = prepare_test_dataset(
        data_root,
        model_name,
        teleop=teleop,
        ablations=model.hparams.get("ablations", None),
        model_idx=model_idx,
        sequence_length=model.hparams.get("sequence_length", 100),
    )

    concat = ConcatDataset(list(test_sets.values()))
    loader = DataLoader(
        concat,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    return loader, len(concat)


def summarize_ms(samples_s):
    arr_ms = 1000.0 * np.asarray(samples_s, dtype=np.float64)
    return float(arr_ms.mean()), float(arr_ms.std(ddof=1)) if arr_ms.size > 1 else 0.0


def summarize_hz_from_s(samples_s):
    arr_s = np.asarray(samples_s, dtype=np.float64)
    arr_hz = 1.0 / arr_s
    return float(arr_hz.mean()), float(arr_hz.std(ddof=1)) if arr_hz.size > 1 else 0.0


@torch.inference_mode()
def benchmark_model(
    model,
    loader,
    device: torch.device,
    warmup_steps: int = 30,
    max_steps: int = 400,
):
    model = model.to(device)
    model.eval()

    if isinstance(model, LitLSTM):
        model.reset_hidden()

    effective_steps = min(max_steps, len(loader))
    if effective_steps <= warmup_steps:
        raise ValueError("Increase MAX_STEPS or reduce WARMUP_STEPS to keep measured steps > 0.")

    # Warm-up
    loader_iter = iter(loader)
    for warmup_idx in range(warmup_steps):
        try:
            batch = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            batch = next(loader_iter)

        batch = move_batch_to_device(batch, device)
        _ = run_model_test_step(model, batch, warmup_idx)

    maybe_sync(device)

    # Timed run
    dataloader_samples_s = []
    transfer_samples_s = []
    inference_samples_s = []

    loader_iter = iter(loader)
    measured_steps = effective_steps - warmup_steps

    for step_idx in tqdm(range(measured_steps), desc=f"{model._get_name()} on {device.type}", leave=False):
        t0 = perf_counter()
        try:
            batch_cpu = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            batch_cpu = next(loader_iter)
        maybe_sync(device)
        t1 = perf_counter()

        batch = move_batch_to_device(batch_cpu, device)
        maybe_sync(device)
        t2 = perf_counter()

        _ = run_model_test_step(model, batch, step_idx)
        maybe_sync(device)
        t3 = perf_counter()

        dataloader_samples_s.append(t1 - t0)
        transfer_samples_s.append(t2 - t1)
        inference_samples_s.append(t3 - t2)

    dataloader_mean, dataloader_std = summarize_ms(dataloader_samples_s)
    transfer_mean, transfer_std = summarize_ms(transfer_samples_s)
    inference_mean, inference_std = summarize_ms(inference_samples_s)
    end_to_end_samples_s = [a + b + c for a, b, c in zip(dataloader_samples_s, transfer_samples_s, inference_samples_s)]
    end_to_end_mean, end_to_end_std = summarize_ms(end_to_end_samples_s)
    end_to_end_hz_mean, end_to_end_hz_std = summarize_hz_from_s(end_to_end_samples_s)

    return {
        "device": device.type,
        "steps": measured_steps,
        "dataloader_ms_mean": dataloader_mean,
        "dataloader_ms_std": dataloader_std,
        "to_device_ms_mean": transfer_mean,
        "to_device_ms_std": transfer_std,
        "inference_ms_mean": inference_mean,
        "inference_ms_std": inference_std,
        "end_to_end_ms_mean": end_to_end_mean,
        "end_to_end_ms_std": end_to_end_std,
        "end_to_end_hz_mean": end_to_end_hz_mean,
        "end_to_end_hz_std": end_to_end_hz_std,
    }

## Load Models

In [ ]:
models = {}
for key, ckpt_dir in CHECKPOINT_DIRS.items():
    if ckpt_dir is None:
        raise ValueError(f"Set CHECKPOINT_DIRS['{key}'] to a valid checkpoint directory.")

    ckpt_dir = Path(ckpt_dir)
    if not ckpt_dir.exists():
        raise FileNotFoundError(f"Checkpoint directory not found: {ckpt_dir}")

    model = load_model(ckpt_dir)
    models[key] = model

list(models.keys())

## Build DataLoaders (Batch Size = 1)

In [ ]:
loaders = {}
sizes = {}

for key, model in models.items():
    loader, n_samples = build_loader_for_model(
        model=model,
        data_root=DATA_ROOT,
        teleop=TELEOP,
        model_idx=MODEL_IDX,
    )
    loaders[key] = loader
    sizes[key] = n_samples

sizes

## Benchmark on CPU and CUDA

In [ ]:
devices = [torch.device("cpu")]
if torch.cuda.is_available():
    devices.append(torch.device("cuda"))

results = []

for device in devices:
    print(f"\n=== Device: {device.type} ===")
    for model_key, model in models.items():
        out = benchmark_model(
            model=model,
            loader=loaders[model_key],
            device=device,
            warmup_steps=WARMUP_STEPS,
            max_steps=MAX_STEPS,
        )
        out["model"] = model_key
        results.append(out)
        print(
            f"{model_key:12s} | "
            f"dataloader: {out['dataloader_ms_mean']:.3f} ± {out['dataloader_ms_std']:.3f} ms | "
            f"to(device): {out['to_device_ms_mean']:.3f} ± {out['to_device_ms_std']:.3f} ms | "
            f"forward: {out['inference_ms_mean']:.3f} ± {out['inference_ms_std']:.3f} ms | "
            f"end-to-end: {out['end_to_end_ms_mean']:.3f} ± {out['end_to_end_ms_std']:.3f} ms | "
            f"Hz: {out['end_to_end_hz_mean']:.3f} ± {out['end_to_end_hz_std']:.3f}"
        )

## Results Table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
if not df.empty:
    df = df[[
        "model",
        "device",
        "steps",
        "dataloader_ms_mean",
        "dataloader_ms_std",
        "to_device_ms_mean",
        "to_device_ms_std",
        "inference_ms_mean",
        "inference_ms_std",
        "end_to_end_ms_mean",
        "end_to_end_ms_std",
        "end_to_end_hz_mean",
        "end_to_end_hz_std",
    ]].sort_values(["device", "model"]).reset_index(drop=True)

    df["dataloader_ms"] = df.apply(
        lambda r: f"{r['dataloader_ms_mean']:.3f} ± {r['dataloader_ms_std']:.3f}", axis=1
    )
    df["to_device_ms"] = df.apply(
        lambda r: f"{r['to_device_ms_mean']:.3f} ± {r['to_device_ms_std']:.3f}", axis=1
    )
    df["inference_ms"] = df.apply(
        lambda r: f"{r['inference_ms_mean']:.3f} ± {r['inference_ms_std']:.3f}", axis=1
    )
    df["end_to_end_ms"] = df.apply(
        lambda r: f"{r['end_to_end_ms_mean']:.3f} ± {r['end_to_end_ms_std']:.3f}", axis=1
    )
    df["end_to_end_Hz"] = df.apply(
        lambda r: f"{r['end_to_end_hz_mean']:.3f} ± {r['end_to_end_hz_std']:.3f}", axis=1
    )

    df = df[[
        "model",
        "device",
        "steps",
        "dataloader_ms",
        "to_device_ms",
        "inference_ms",
        "end_to_end_ms",
        "end_to_end_Hz",
    ]]

df